# Hands-On Lab: Comprehensive Evaluation Pipeline
# Credit Card Fraud Detection

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 4 hours  
**Week 7 | Notebook 12 | Hands-On Lab**

---

## Real-World Scenario

You are a machine learning engineer at a fintech company. The fraud team gives you a dataset of 10,000 transactions (3% fraud). Your job: **build, evaluate, and compare four classifiers**, then write a recommendation memo to the business.

The twist: your manager asks three different questions — each one implying a different metric, a different winning model, and a different deployment decision.

**By the end of this lab you will understand why "best model" is meaningless without a business context.**

---

## Learning Objectives

| # | Objective |
|---|---|
| 1 | Implement 5-fold Stratified CV from scratch |
| 2 | Build full confusion matrix reports with per-class percentages |
| 3 | Compare ROC and Precision-Recall curves across 4 models |
| 4 | Evaluate model calibration and understand its business impact |
| 5 | Show how metric choice changes model rankings |
| 6 | Apply McNemar's test and Bonferroni correction |
| 7 | Write a structured model recommendation backed by data |

## Section 0: Setup

In [ ]:
# ── All Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Classifiers
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier

# Preprocessing and utilities
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold

# Metrics
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, precision_score, recall_score,
    roc_curve, precision_recall_curve
)

# Calibration
from sklearn.calibration import CalibrationDisplay

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')
np.random.seed(42)

print('All imports successful.')
print('NumPy:', np.__version__)
print('Pandas:', pd.__version__)

In [ ]:
# ── Create Synthetic Credit Card Fraud Dataset ────────────────────────────────
# 10,000 transactions, 3% fraud (300 fraudulent, 9700 legitimate)
# Features: amount, time_since_last_txn, num_txns_7d, credit_score,
#           account_age_days, merchant_risk_score, distance_from_home,
#           is_international, is_night, card_type

np.random.seed(42)
N_TOTAL = 10_000
FRAUD_RATE = 0.03
n_fraud = int(N_TOTAL * FRAUD_RATE)     # 300
n_legit = N_TOTAL - n_fraud             # 9700

# ── Legitimate transactions ──────────────────────────────────────────────────
legit = {
    'amount'              : np.random.exponential(scale=85,   size=n_legit),
    'time_since_last_txn' : np.random.exponential(scale=30,   size=n_legit),  # hours
    'num_txns_7d'         : np.random.poisson(lam=4,          size=n_legit),
    'credit_score'        : np.random.normal(loc=710, scale=55, size=n_legit),
    'account_age_days'    : np.random.exponential(scale=1200, size=n_legit),
    'merchant_risk_score' : np.random.beta(a=2,  b=8,         size=n_legit),
    'distance_from_home'  : np.random.exponential(scale=15,   size=n_legit),  # km
    'is_international'    : (np.random.rand(n_legit) < 0.04).astype(int),
    'is_night'            : (np.random.rand(n_legit) < 0.12).astype(int),
    'card_type'           : np.random.choice([0, 1, 2], size=n_legit, p=[0.55, 0.35, 0.10]),
}

# ── Fraudulent transactions (shifted distributions) ──────────────────────────
fraud = {
    'amount'              : np.random.exponential(scale=400,  size=n_fraud),  # larger
    'time_since_last_txn' : np.random.exponential(scale=1.5,  size=n_fraud),  # rapid
    'num_txns_7d'         : np.random.poisson(lam=18,         size=n_fraud),  # many txns
    'credit_score'        : np.random.normal(loc=590, scale=85, size=n_fraud),
    'account_age_days'    : np.random.exponential(scale=150,  size=n_fraud),  # newer
    'merchant_risk_score' : np.random.beta(a=6,  b=4,         size=n_fraud),  # riskier
    'distance_from_home'  : np.random.exponential(scale=250,  size=n_fraud),  # far away
    'is_international'    : (np.random.rand(n_fraud) < 0.45).astype(int),
    'is_night'            : (np.random.rand(n_fraud) < 0.50).astype(int),
    'card_type'           : np.random.choice([0, 1, 2], size=n_fraud, p=[0.25, 0.50, 0.25]),
}

# ── Assemble DataFrame ───────────────────────────────────────────────────────
feature_cols = list(legit.keys())
df_legit  = pd.DataFrame(legit)
df_fraud  = pd.DataFrame(fraud)
df_legit['fraud'] = 0
df_fraud['fraud'] = 1

df = pd.concat([df_legit, df_fraud], ignore_index=True)

# Clip unrealistic values
df['credit_score']  = df['credit_score'].clip(300, 850)
df['amount']        = df['amount'].clip(0.01, 10000)
df['account_age_days'] = df['account_age_days'].clip(1, 10000)

# Shuffle the dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

BASE_RATE = df['fraud'].mean()

print(f'Dataset shape       : {df.shape}')
print(f'Fraud count         : {df["fraud"].sum()}')
print(f'Fraud rate (base)   : {BASE_RATE:.2%}')
print(f'\nFeatures: {feature_cols}')
df.describe().round(2)

## Section 1: 5-Fold Stratified CV From Scratch

Stratified means each fold preserves the class ratio (~3% fraud). We implement this without using `sklearn.model_selection.StratifiedKFold` directly.

In [ ]:
# ── Stratified K-Fold CV Implementation From Scratch ─────────────────────────
# Step-by-step: group by class, divide each class into k chunks, interleave.

def stratified_kfold_cv(model, X, y, k=5, scoring_funcs=None, random_state=42):
    """
    Perform k-fold stratified cross-validation from scratch.

    Algorithm:
    1. Separate indices by class
    2. Shuffle indices within each class
    3. Split each class indices into k roughly equal chunks
    4. For fold i: test set = chunk i from each class, train = remaining
    5. Fit model on train, compute metrics on test

    Parameters
    ----------
    model         : sklearn-compatible model with fit() and predict_proba()
    X             : feature array
    y             : label array (binary: 0/1)
    k             : number of folds
    scoring_funcs : dict of {metric_name: callable(y_true, y_pred_or_proba)}
    random_state  : random seed for shuffling

    Returns
    -------
    dict of {metric_name: np.array of k scores}
    """
    if scoring_funcs is None:
        scoring_funcs = {
            'accuracy': lambda yt, yp, ypr: accuracy_score(yt, yp),
            'f1'      : lambda yt, yp, ypr: f1_score(yt, yp, zero_division=0),
            'auc_roc' : lambda yt, yp, ypr: roc_auc_score(yt, ypr),
            'ap'      : lambda yt, yp, ypr: average_precision_score(yt, ypr),
        }

    X = np.asarray(X)
    y = np.asarray(y)
    rng = np.random.RandomState(random_state)

    # Group indices by class and shuffle within each class
    classes  = np.unique(y)
    idx_by_class = {}
    for c in classes:
        idx_c = np.where(y == c)[0]
        rng.shuffle(idx_c)
        idx_by_class[c] = idx_c

    # Split each class into k chunks
    folds = [[] for _ in range(k)]    # folds[i] = list of indices in fold i
    for c, idx_c in idx_by_class.items():
        chunk_size = len(idx_c) // k
        for fold_i in range(k):
            start = fold_i * chunk_size
            end   = start + chunk_size if fold_i < k - 1 else len(idx_c)  # last fold gets remainder
            folds[fold_i].extend(idx_c[start:end].tolist())

    # Run CV: for each fold, train on the other k-1, test on this fold
    all_scores = {name: [] for name in scoring_funcs}

    for fold_i in range(k):
        test_idx  = np.array(folds[fold_i])
        train_idx = np.array([i for j, fold in enumerate(folds) if j != fold_i for i in fold])

        X_tr, y_tr = X[train_idx], y[train_idx]
        X_te, y_te = X[test_idx],  y[test_idx]

        model.fit(X_tr, y_tr)
        y_pred   = model.predict(X_te)
        y_proba  = model.predict_proba(X_te)[:, 1]  # probability of class 1 (fraud)

        for name, func in scoring_funcs.items():
            score = func(y_te, y_pred, y_proba)
            all_scores[name].append(score)

    return {name: np.array(scores) for name, scores in all_scores.items()}


print('stratified_kfold_cv() defined.')
print('Ready to run on all 4 models.')

In [ ]:
# ── Define Models and Run 5-Fold CV ──────────────────────────────────────────
# Use pipelines to avoid data leakage: scaler fit inside each CV fold

X_all = df[feature_cols].values
y_all = df['fraud'].values

models = {
    'LogReg'      : Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=42))]),
    'k-NN'        : Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=5))]),
    'GaussianNB'  : Pipeline([('scaler', StandardScaler()), ('clf', GaussianNB())]),
    'DecisionTree': Pipeline([('scaler', StandardScaler()), ('clf', DecisionTreeClassifier(max_depth=5, random_state=42))]),
}

# Score functions: each takes (y_true, y_pred, y_proba)
scoring_funcs = {
    'accuracy' : lambda yt, yp, ypr: accuracy_score(yt, yp),
    'f1'       : lambda yt, yp, ypr: f1_score(yt, yp, zero_division=0),
    'auc_roc'  : lambda yt, yp, ypr: roc_auc_score(yt, ypr),
    'ap'       : lambda yt, yp, ypr: average_precision_score(yt, ypr),
}

# --- Run CV for each model ---
cv_results = {}   # model_name -> {metric -> array of 5 scores}

print('Running 5-fold stratified CV (from scratch)...')
for name, model in models.items():
    scores = stratified_kfold_cv(model, X_all, y_all, k=5, scoring_funcs=scoring_funcs)
    cv_results[name] = scores
    print(f'  {name:15s} done.  AUC={scores["auc_roc"].mean():.4f}  AP={scores["ap"].mean():.4f}')

print('\nCV complete.')

In [ ]:
# ── CV Results Table: Fold-by-Fold Scores ────────────────────────────────────
# Build a comprehensive display table: mean ± std per metric per model

model_names  = list(models.keys())
metric_names = list(scoring_funcs.keys())

# --- Fold-by-fold table for each model ---
print('Fold-by-Fold CV Scores')
print('=' * 75)

rows = []
for name in model_names:
    for fold_i in range(5):
        row = {'Model': name, 'Fold': fold_i + 1}
        for m in metric_names:
            row[m] = cv_results[name][m][fold_i]
        rows.append(row)

fold_df = pd.DataFrame(rows)
print(fold_df.to_string(index=False, float_format='{:.4f}'.format))

# --- Summary table: mean ± std ---
print('\n\nCV Summary: Mean ± Std across 5 folds')
print('=' * 75)

summary_rows = []
for name in model_names:
    row = {'Model': name}
    for m in metric_names:
        arr  = cv_results[name][m]
        row[f'{m}_mean'] = arr.mean()
        row[f'{m}_std']  = arr.std()
        row[f'{m}']      = f'{arr.mean():.4f} ± {arr.std():.4f}'
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display_cols = ['Model'] + metric_names
print(summary_df[display_cols].to_string(index=False))

In [ ]:
# ── Train Final Models on Full Dataset for Holdout Evaluation ─────────────────
# Use 80% for training final models, 20% as holdout test set (stratified)
# This test set is shared by all models (required for McNemar's test later)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all,
    test_size=0.20, random_state=42, stratify=y_all
)

print(f'Training set : {X_train.shape[0]} samples ({y_train.mean():.2%} fraud)')
print(f'Test set     : {X_test.shape[0]}  samples ({y_test.mean():.2%} fraud)')

# Train all models and collect predictions & probabilities on the test set
test_preds  = {}    # binary predictions
test_probas = {}    # predicted probabilities of fraud
final_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    test_preds[name]  = model.predict(X_test)
    test_probas[name] = model.predict_proba(X_test)[:, 1]
    final_models[name] = model

print('\nAll models trained. Predictions collected on shared test set.')

## Section 2: Full Confusion Matrix Report

In [ ]:
# ── Confusion Matrix Heatmaps: 2×2 Subplot Grid ──────────────────────────────
# Each cell shows: count AND percentage of actual class (row-normalized)

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle('Confusion Matrices: Credit Card Fraud Detection\n(counts + % of actual class)',
             fontsize=14, fontweight='bold')

cell_labels_map = {
    (0, 0): 'TN\n(Legit → Legit)',
    (0, 1): 'FP\n(Legit → Fraud)',
    (1, 0): 'FN\n(Fraud → Legit)',
    (1, 1): 'TP\n(Fraud → Fraud)'
}

cm_summary = {}   # for the summary table

for ax, name in zip(axes.flat, model_names):
    cm = confusion_matrix(y_test, test_preds[name])
    tn, fp, fn, tp = cm.ravel()
    cm_summary[name] = {'TP': tp, 'FP': fp, 'TN': tn, 'FN': fn}

    # Row-normalize to get percentage of actual class
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    # Build annotation strings: count + %
    annot = np.empty_like(cm, dtype=object)
    for i in range(2):
        for j in range(2):
            label = cell_labels_map[(i, j)]
            annot[i, j] = f'{cm[i,j]}\n({cm_pct[i,j]:.1f}% of actual)\n{label}'

    sns.heatmap(
        cm, ax=ax,
        annot=annot, fmt='', cmap='Blues',
        xticklabels=['Predicted: Legit', 'Predicted: Fraud'],
        yticklabels=['Actual: Legit', 'Actual: Fraud'],
        cbar=False, linewidths=1, linecolor='gray'
    )
    acc = accuracy_score(y_test, test_preds[name])
    f1  = f1_score(y_test, test_preds[name], zero_division=0)
    ax.set_title(f'{name}\nAccuracy={acc:.4f}  |  F1={f1:.4f}', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Summary Table: TP, FP, TN, FN for All 4 Models ───────────────────────────
# Compute derived metrics: precision, recall, specificity

rows = []
for name in model_names:
    d = cm_summary[name]
    tp, fp, tn, fn = d['TP'], d['FP'], d['TN'], d['FN']
    precision   = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall      = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    rows.append({
        'Model'      : name,
        'TP'         : tp,
        'FP'         : fp,
        'TN'         : tn,
        'FN'         : fn,
        'Precision'  : f'{precision:.4f}',
        'Recall'     : f'{recall:.4f}',
        'Specificity': f'{specificity:.4f}',
        'F1'         : f'{f1:.4f}'
    })

cm_df = pd.DataFrame(rows)
print('Confusion Matrix Summary — All 4 Models on Test Set')
print('=' * 80)
print(cm_df.to_string(index=False))
print(f'\nTest set: {y_test.sum()} fraud cases out of {len(y_test)} total')
print(f'Business note: FN = missed frauds (financial loss); FP = false alarms (customer friction)')

## Section 3: ROC Curves

In [ ]:
# ── ROC Curves for All 4 Models on One Axes ───────────────────────────────────
# Also shows the default threshold=0.5 operating point for each model

fig, ax = plt.subplots(figsize=(9, 7))

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
roc_aucs = {}

for color, name in zip(colors, model_names):
    fpr, tpr, thresholds = roc_curve(y_test, test_probas[name])
    auc_val = roc_auc_score(y_test, test_probas[name])
    roc_aucs[name] = auc_val

    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name}  (AUC = {auc_val:.4f})')

    # Mark the default threshold=0.5 operating point on the ROC curve
    # Find the threshold closest to 0.5
    thresh_idx = np.argmin(np.abs(thresholds - 0.5))
    ax.scatter(fpr[thresh_idx], tpr[thresh_idx], color=color,
               s=100, zorder=5, marker='D', edgecolors='black', linewidths=1)

# Random baseline diagonal
ax.plot([0, 1], [0, 1], 'k--', lw=1.5, alpha=0.6, label='Random baseline (AUC=0.50)')

# Shade FPR < 0.10 region (partial AUC zone)
ax.axvline(x=0.10, color='gray', linestyle=':', lw=1.5, alpha=0.7, label='FPR=0.10 boundary')
ax.fill_betweenx([0, 1], 0, 0.10, alpha=0.05, color='green')
ax.text(0.05, 0.15, 'Partial AUC\n(FPR<0.10)', ha='center', va='bottom', fontsize=9, color='green')

ax.set_xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
ax.set_ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=12)
ax.set_title('ROC Curves — Credit Card Fraud Detection\n◇ = operating point at default threshold 0.5', fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)

plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nAUC-ROC scores:', {k: round(v, 4) for k, v in roc_aucs.items()})

In [ ]:
# ── Partial AUC Comparison (FPR < 0.10) ──────────────────────────────────────
# In fraud detection, we often care only about the low-FPR regime:
# "At most 10% of legitimate transactions are flagged — how many frauds do we catch?"
# Partial AUC captures this operationally relevant region.

from sklearn.metrics import roc_auc_score

fpr_limit = 0.10

print(f'Partial AUC Comparison (FPR < {fpr_limit})')
print('=' * 55)
print(f'{"Model":15s}  {"Full AUC":>10s}  {"Partial AUC":>12s}  {"Ratio":>8s}')
print('-' * 55)

partial_aucs = {}
for name in model_names:
    fpr, tpr, _ = roc_curve(y_test, test_probas[name])

    # Interpolate the TPR at FPR = fpr_limit for a clean boundary
    mask = fpr <= fpr_limit
    # Trapezoidal integration under the curve in the partial region
    partial_auc = np.trapz(tpr[mask], fpr[mask])
    full_auc    = roc_aucs[name]
    ratio       = partial_auc / fpr_limit  # normalized to [0,1] scale

    partial_aucs[name] = partial_auc
    print(f'{name:15s}  {full_auc:10.4f}  {partial_auc:12.4f}  {ratio:8.4f}')

best_partial = max(partial_aucs, key=partial_aucs.get)
print(f'\nBest model in the low-FPR regime (FPR<{fpr_limit}): {best_partial}')
print('Note: partial AUC winner may differ from full AUC winner!')

## Section 4: Precision-Recall Curves

In [ ]:
# ── Precision-Recall Curves for All 4 Models ─────────────────────────────────
# Includes: horizontal base rate baseline, iso-F1 lines, AP in legend

fig, ax = plt.subplots(figsize=(9, 7))

avg_precisions = {}

for color, name in zip(colors, model_names):
    precision, recall, thresholds_pr = precision_recall_curve(y_test, test_probas[name])
    ap = average_precision_score(y_test, test_probas[name])
    avg_precisions[name] = ap

    ax.plot(recall, precision, color=color, lw=2, label=f'{name}  (AP = {ap:.4f})')

    # Mark optimal threshold (maximizes F1)
    # Avoid division by zero: only where recall > 0 and precision > 0
    f1_scores_pr = np.where(
        (precision[:-1] + recall[:-1]) > 0,
        2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1]),
        0
    )
    best_idx     = np.argmax(f1_scores_pr)
    best_recall  = recall[best_idx]
    best_prec    = precision[best_idx]
    best_thresh  = thresholds_pr[best_idx]
    ax.scatter(best_recall, best_prec, color=color, s=120, zorder=5,
               marker='*', edgecolors='black', linewidths=0.5)

# Horizontal baseline: random classifier at base rate
ax.axhline(y=BASE_RATE, color='gray', linestyle='--', lw=1.5, alpha=0.8,
           label=f'No-skill baseline (precision = {BASE_RATE:.2%})')

# Iso-F1 lines: curves where F1 = constant
recall_grid = np.linspace(0.01, 1.0, 300)
for f1_target in [0.1, 0.2, 0.4]:
    # P = F1 * R / (2R - F1)  →  derived from F1 = 2PR/(P+R)
    iso_prec = np.where(
        (2 * recall_grid - f1_target) > 0,
        f1_target * recall_grid / (2 * recall_grid - f1_target),
        np.nan
    )
    valid = (iso_prec >= 0) & (iso_prec <= 1)
    ax.plot(recall_grid[valid], iso_prec[valid], 'k-', lw=0.8, alpha=0.4)
    # Label at recall=0.9
    idx_label = np.searchsorted(recall_grid, 0.85)
    if valid[idx_label]:
        ax.text(recall_grid[idx_label], iso_prec[idx_label], f'F1={f1_target}',
                fontsize=8, color='gray', alpha=0.8)

ax.set_xlabel('Recall (Sensitivity)', fontsize=12)
ax.set_ylabel('Precision (Positive Predictive Value)', fontsize=12)
ax.set_title('Precision-Recall Curves — Credit Card Fraud Detection\n'
             '★ = threshold maximising F1  |  dashed = iso-F1 lines',
             fontsize=12, fontweight='bold')
ax.legend(loc='upper right', fontsize=9)
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.05)

plt.tight_layout()
plt.savefig('pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('\nAverage Precision (AP):', {k: round(v, 4) for k, v in avg_precisions.items()})

In [ ]:
# ── Optimal Threshold per Model (Maximise F1) ─────────────────────────────────
# Finding the threshold that maximizes F1 is a business-critical step:
# the default 0.5 is rarely optimal for imbalanced fraud data.

print('Optimal Threshold per Model (maximising F1 score)')
print('=' * 75)
print(f'{"Model":15s}  {"Opt. Threshold":>15s}  {"Precision":>10s}  {"Recall":>8s}  {"F1":>8s}')
print('-' * 65)

optimal_thresholds = {}
for name in model_names:
    precision_arr, recall_arr, thresh_arr = precision_recall_curve(y_test, test_probas[name])

    # Compute F1 at each threshold (exclude last point where recall=0)
    f1_arr = np.where(
        (precision_arr[:-1] + recall_arr[:-1]) > 0,
        2 * precision_arr[:-1] * recall_arr[:-1] / (precision_arr[:-1] + recall_arr[:-1]),
        0
    )

    best_idx      = np.argmax(f1_arr)
    best_threshold = thresh_arr[best_idx]
    best_prec      = precision_arr[best_idx]
    best_rec       = recall_arr[best_idx]
    best_f1        = f1_arr[best_idx]

    optimal_thresholds[name] = best_threshold
    print(f'{name:15s}  {best_threshold:15.4f}  {best_prec:10.4f}  {best_rec:8.4f}  {best_f1:8.4f}')

print(f'\nNote: default threshold = 0.5 is suboptimal for 3% fraud rate.')
print(f'Lowering the threshold increases recall (catch more fraud) at cost of precision (more false alarms).')

## Section 5: Calibration Plots

A model is **well-calibrated** if its predicted probability of fraud matches the actual fraction of fraudulent transactions. 

**Why it matters for threshold selection**: if GaussianNB predicts P(fraud) = 0.9 for cases that are only 40% fraud, then setting threshold = 0.5 will catch everything above 0.9 — but you're applying the wrong threshold. A calibrated model lets you say "flag transactions above P=0.2" and trust that 20% of those are actually fraud.

In [ ]:
# ── Calibration Reliability Diagrams ─────────────────────────────────────────
# CalibrationDisplay shows: predicted probability bins vs observed frequency
# Perfect calibration → diagonal line (predicted = actual)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Calibration Reliability Diagrams\n'
             'Diagonal = perfect calibration | Above diagonal = underconfident | Below = overconfident',
             fontsize=12, fontweight='bold')

# Left: individual plots overlaid
ax_overlay = axes[0]
for color, name in zip(colors, model_names):
    CalibrationDisplay.from_predictions(
        y_test, test_probas[name],
        n_bins=10, name=name,
        ax=ax_overlay, color=color
    )

ax_overlay.set_title('All Models Overlaid', fontsize=11)
ax_overlay.legend(loc='upper left', fontsize=9)

# Right: histogram of predicted probabilities (shows overconfidence)
ax_hist = axes[1]
for color, name in zip(colors, model_names):
    ax_hist.hist(
        test_probas[name], bins=30, alpha=0.5, color=color,
        label=name, density=True, histtype='stepfilled'
    )
ax_hist.axvline(BASE_RATE, color='black', ls='--', lw=2, label=f'Base rate ({BASE_RATE:.2%})')
ax_hist.set_xlabel('Predicted P(fraud)', fontsize=11)
ax_hist.set_ylabel('Density', fontsize=11)
ax_hist.set_title('Distribution of Predicted Fraud Probabilities', fontsize=11)
ax_hist.legend(fontsize=9)

plt.tight_layout()
plt.savefig('calibration_plots.png', dpi=150, bbox_inches='tight')
plt.show()

print('Calibration discussion:')
print('  LogReg    — typically well-calibrated (maximises log-loss = calibration)')
print('  k-NN      — often overconfident near extremes (predictions cluster near 0 or 1)')
print('  GaussianNB — frequently miscalibrated; assumes feature independence → extreme probs')
print('  DecTree   — poorly calibrated at leaves (predictions = 0.0 or 1.0 for deep trees)')

## Section 6: Metric Choice Changes Model Selection

**This is the most important lesson of the lab.**

The metric you report determines which model appears to "win". The fraud team and the engineering team may legitimately want different things — and your job is to be honest about the trade-offs.

In [ ]:
# ── Compute All Metrics on the Test Set ──────────────────────────────────────
# accuracy, F1, AUC-ROC, AP, precision@recall=0.5

def precision_at_recall(y_true, y_proba, target_recall=0.50):
    """Return precision at the threshold where recall >= target_recall."""
    precision_arr, recall_arr, _ = precision_recall_curve(y_true, y_proba)
    # Find threshold where recall first drops to or below target_recall
    # precision_recall_curve returns (precision, recall) in decreasing recall order
    mask = recall_arr >= target_recall
    if not np.any(mask):
        return 0.0
    return float(precision_arr[mask][-1])   # last precision value where recall >= target


metrics_all = {}
for name in model_names:
    preds  = test_preds[name]
    probas = test_probas[name]
    metrics_all[name] = {
        'Accuracy'            : accuracy_score(y_test, preds),
        'F1'                  : f1_score(y_test, preds, zero_division=0),
        'AUC-ROC'             : roc_auc_score(y_test, probas),
        'AP (Avg Precision)'  : average_precision_score(y_test, probas),
        'Precision@Recall=0.5': precision_at_recall(y_test, probas, target_recall=0.5),
    }

metrics_df = pd.DataFrame(metrics_all).T
metrics_df.index.name = 'Model'

print('Test Set Metrics — All 4 Models')
print('=' * 75)
print(metrics_df.round(4).to_string())

In [ ]:
# ── Ranking Table: Which Model Wins Under Each Metric? ────────────────────────
# The key insight: the winner depends entirely on what you measure

metric_cols = list(metrics_all[model_names[0]].keys())

# Rank models by each metric (rank 1 = best)
rank_data = {}
for metric in metric_cols:
    vals  = {name: metrics_all[name][metric] for name in model_names}
    sorted_names = sorted(vals, key=vals.get, reverse=True)    # descending: higher = better
    ranks = {name: sorted_names.index(name) + 1 for name in model_names}
    rank_data[metric] = ranks

rank_df = pd.DataFrame(rank_data, index=model_names)
rank_df.index.name = 'Model'

print('Model Rankings by Metric (1 = best)\n')
print(rank_df.to_string())

print('\n#1 model by each metric:')
for metric in metric_cols:
    winner = min(rank_data[metric], key=rank_data[metric].get)
    score  = metrics_all[winner][metric]
    print(f'  {metric:30s} → {winner}  ({score:.4f})')

print('\nKey insight: there is no single "best" model — it depends on the metric!')

In [ ]:
# ── Business Scenario Analysis ────────────────────────────────────────────────
# Three realistic business questions → three different winners

print('Business Scenario Analysis\n')
print('=' * 75)

# ── Scenario 1: "Catch 80% of fraud, even if some false alarms" ───────────────
# Optimize: recall >= 0.80 with minimum false positives
print('SCENARIO 1: Regulatory requirement — catch at least 80% of fraud')
print('  Metric: Recall ≥ 0.80, then minimize FPR')
print()

scenario1 = {}
for name in model_names:
    fpr_arr, tpr_arr, thresh_arr = roc_curve(y_test, test_probas[name])
    # Find threshold where TPR (recall) >= 0.80
    mask    = tpr_arr >= 0.80
    if not np.any(mask):
        scenario1[name] = {'recall': 0, 'fpr': 1, 'threshold': None}
        continue
    # Among those, find the one with lowest FPR (first point where mask is True, which has lowest FPR)
    first_idx = np.where(mask)[0][0]
    scenario1[name] = {
        'recall'    : tpr_arr[first_idx],
        'fpr'       : fpr_arr[first_idx],
        'threshold' : thresh_arr[first_idx] if first_idx < len(thresh_arr) else 0
    }
    print(f'  {name:15s}: recall={tpr_arr[first_idx]:.4f}  FPR={fpr_arr[first_idx]:.4f}  threshold={scenario1[name]["threshold"]:.4f}')

best_s1 = min(scenario1, key=lambda n: scenario1[n]['fpr'])
print(f'  → WINNER: {best_s1} (lowest false positive rate while catching ≥80% fraud)')

# ── Scenario 2: "Do not exceed 5% FPR on legitimate transactions" ─────────────
print()
print('SCENARIO 2: Customer experience — no more than 5% of legit transactions flagged')
print('  Metric: FPR ≤ 0.05, then maximize recall')
print()

scenario2 = {}
for name in model_names:
    fpr_arr, tpr_arr, _ = roc_curve(y_test, test_probas[name])
    mask    = fpr_arr <= 0.05
    if not np.any(mask):
        scenario2[name] = {'recall': 0, 'fpr': 1}
        continue
    best_tpr = tpr_arr[mask].max()     # maximize recall within FPR constraint
    best_fpr = fpr_arr[mask][np.argmax(tpr_arr[mask])]
    scenario2[name] = {'recall': best_tpr, 'fpr': best_fpr}
    print(f'  {name:15s}: recall={best_tpr:.4f}  FPR={best_fpr:.4f}')

best_s2 = max(scenario2, key=lambda n: scenario2[n]['recall'])
print(f'  → WINNER: {best_s2} (highest recall at FPR ≤ 5%)')

# ── Scenario 3: "Best overall ranking quality" ────────────────────────────────
print()
print('SCENARIO 3: Model selection for a risk scoring system — ranking quality matters most')
print('  Metric: AUC-ROC (overall ranking of fraud probability)')
print()

for name in model_names:
    auc = roc_aucs[name]
    print(f'  {name:15s}: AUC-ROC = {auc:.4f}')

best_s3 = max(roc_aucs, key=roc_aucs.get)
print(f'  → WINNER: {best_s3}')

print()
print('SUMMARY: Three scenarios, potentially three different winners.')
print('Always define the business objective BEFORE evaluating models.')

In [ ]:
# ── Visualise Metric Rankings as a Heatmap ────────────────────────────────────
# Visual summary of which model ranks best on each metric

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: raw metric values
ax1 = axes[0]
metrics_vals = pd.DataFrame(metrics_all).T
sns.heatmap(
    metrics_vals, ax=ax1, annot=True, fmt='.4f',
    cmap='YlOrRd', linewidths=0.5,
    cbar_kws={'label': 'Metric value'}
)
ax1.set_title('Test Set Metric Values\n(higher = better for all)', fontsize=11, fontweight='bold')
ax1.set_xlabel('')
ax1.tick_params(axis='x', rotation=30)

# Right: ranks (1=best, 4=worst)
ax2 = axes[1]
sns.heatmap(
    rank_df, ax=ax2, annot=True, fmt='d',
    cmap='RdYlGn_r', linewidths=0.5, vmin=1, vmax=4,
    cbar_kws={'label': 'Rank (1=best, 4=worst)'}
)
ax2.set_title('Model Rankings by Metric\n(1 = best performer)', fontsize=11, fontweight='bold')
ax2.set_xlabel('')
ax2.tick_params(axis='x', rotation=30)

plt.suptitle('Metric Choice Determines Which Model Wins', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('metric_rankings.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 7: McNemar's Test — Are Performance Differences Statistically Real?

In [ ]:
# ── McNemar's Test: From-Scratch Implementation ───────────────────────────────
# Same implementation as Notebook 11, repeated here for completeness

def mcnemar_test(y_true, preds_a, preds_b, continuity_correction=True):
    """
    McNemar's test for comparing two classifiers on the same test set.
    Returns contingency table counts, chi-squared statistic, and p-value.
    """
    y_true  = np.asarray(y_true)
    preds_a = np.asarray(preds_a)
    preds_b = np.asarray(preds_b)

    correct_a = (preds_a == y_true)
    correct_b = (preds_b == y_true)

    # 2×2 contingency table
    n11 = int(np.sum( correct_a &  correct_b))   # both correct
    n10 = int(np.sum( correct_a & ~correct_b))   # A correct, B wrong
    n01 = int(np.sum(~correct_a &  correct_b))   # B correct, A wrong
    n00 = int(np.sum(~correct_a & ~correct_b))   # both wrong

    # Test statistic with optional Yates' continuity correction
    denom = n01 + n10
    if denom == 0:
        return dict(n11=n11, n10=n10, n01=n01, n00=n00, chi2=0.0, p_value=1.0)

    if continuity_correction:
        chi2 = (abs(n01 - n10) - 1.0) ** 2 / denom
    else:
        chi2 = (n01 - n10) ** 2 / denom

    # p-value from chi-squared distribution with 1 degree of freedom
    p_value = stats.chi2.sf(chi2, df=1)

    return dict(n11=n11, n10=n10, n01=n01, n00=n00,
                chi2=round(chi2, 4), p_value=round(p_value, 6))


# --- Run all C(4,2)=6 pairwise comparisons ---
n_models     = len(model_names)
pval_matrix  = np.ones((n_models, n_models))

print('Pairwise McNemar Tests — All 6 Model Pairs')
print('=' * 70)
print(f'{"Model A":15s}  {"Model B":15s}  {"n10":>5s}  {"n01":>5s}  {"χ²":>8s}  {"p-val":>10s}')
print('-' * 62)

mcnemar_rows = []
for i in range(n_models):
    for j in range(i + 1, n_models):
        na, nb  = model_names[i], model_names[j]
        res     = mcnemar_test(y_test, test_preds[na], test_preds[nb])
        pval_matrix[i, j] = res['p_value']
        pval_matrix[j, i] = res['p_value']
        mcnemar_rows.append({'Model A': na, 'Model B': nb, **res})
        print(f'{na:15s}  {nb:15s}  {res["n10"]:5d}  {res["n01"]:5d}  {res["chi2"]:8.4f}  {res["p_value"]:10.6f}')

mcnemar_df = pd.DataFrame(mcnemar_rows)
print(f'\n(n10 = Model A correct, B wrong | n01 = B correct, A wrong)')

In [ ]:
# ── Bonferroni Correction + p-value Heatmap ───────────────────────────────────
# k = C(4,2) = 6 tests → Bonferroni α = 0.05/6 ≈ 0.0083

n_pairs         = len(mcnemar_df)     # 6
alpha_raw       = 0.05
alpha_bonf      = alpha_raw / n_pairs

mcnemar_df['Sig_raw']       = mcnemar_df['p_value'] < alpha_raw
mcnemar_df['Sig_Bonferroni'] = mcnemar_df['p_value'] < alpha_bonf
mcnemar_df['p_bonferroni']  = (mcnemar_df['p_value'] * n_pairs).clip(upper=1.0)

print(f'Bonferroni Correction: k={n_pairs} tests  →  α_corrected = {alpha_bonf:.6f}')
print(f'Significant pairs (raw α={alpha_raw}):          {mcnemar_df["Sig_raw"].sum()}')
print(f'Significant pairs (Bonferroni α={alpha_bonf:.4f}): {mcnemar_df["Sig_Bonferroni"].sum()}')
print()
print(mcnemar_df[['Model A', 'Model B', 'p_value', 'Sig_raw', 'p_bonferroni', 'Sig_Bonferroni']].to_string(index=False))

# --- Heatmap of p-values ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('McNemar\'s Test: Pairwise p-values (Lab Dataset)', fontsize=13, fontweight='bold')

mask_diag = np.eye(n_models, dtype=bool)

def build_annot(pval_mat, alpha):
    ann = np.full(pval_mat.shape, '', dtype=object)
    for i in range(n_models):
        for j in range(n_models):
            if i != j:
                p = pval_mat[i, j]
                star = '*' if p < alpha else ''
                ann[i, j] = f'{p:.3f}{star}'
    return ann

# Raw p-values
sns.heatmap(pval_matrix, ax=axes[0], mask=mask_diag,
            annot=build_annot(pval_matrix, alpha_raw), fmt='',
            cmap='RdYlGn_r', vmin=0, vmax=0.1,
            xticklabels=model_names, yticklabels=model_names,
            linewidths=1, linecolor='gray')
axes[0].set_title(f'Raw p-values (α={alpha_raw})\n* = significant', fontsize=10)

# Bonferroni-corrected
bonf_mat = np.clip(pval_matrix * n_pairs, 0, 1)
np.fill_diagonal(bonf_mat, 1.0)
sns.heatmap(bonf_mat, ax=axes[1], mask=mask_diag,
            annot=build_annot(bonf_mat, alpha_raw), fmt='',
            cmap='RdYlGn_r', vmin=0, vmax=1,
            xticklabels=model_names, yticklabels=model_names,
            linewidths=1, linecolor='gray')
axes[1].set_title(f'Bonferroni-corrected p-values\n(α={alpha_bonf:.4f}, * = still significant)', fontsize=10)

plt.tight_layout()
plt.savefig('mcnemar_heatmap_lab.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 8: Summary Report

In [ ]:
# ── Final Summary Table ───────────────────────────────────────────────────────
# Combine all metrics + CV results + overall ranking

print('FINAL EVALUATION REPORT — CREDIT CARD FRAUD DETECTION')
print('=' * 80)
print(f'Dataset: 10,000 transactions | Fraud rate: {BASE_RATE:.2%} | Test set: 2,000 samples')
print()

# Build the comprehensive summary
final_rows = []
for name in model_names:
    cv_auc   = cv_results[name]['auc_roc']
    cv_ap    = cv_results[name]['ap']
    cv_f1    = cv_results[name]['f1']

    d = cm_summary[name]
    tp, fp, tn, fn = d['TP'], d['FP'], d['TN'], d['FN']

    final_rows.append({
        'Model'           : name,
        'CV AUC (mean±std)': f'{cv_auc.mean():.4f}±{cv_auc.std():.4f}',
        'CV AP (mean±std)' : f'{cv_ap.mean():.4f}±{cv_ap.std():.4f}',
        'CV F1 (mean±std)' : f'{cv_f1.mean():.4f}±{cv_f1.std():.4f}',
        'Test Accuracy'    : f'{metrics_all[name]["Accuracy"]:.4f}',
        'Test AUC-ROC'     : f'{metrics_all[name]["AUC-ROC"]:.4f}',
        'Test AP'          : f'{metrics_all[name]["AP (Avg Precision)"]:.4f}',
        'Test F1'          : f'{metrics_all[name]["F1"]:.4f}',
        'TP (fraud caught)': tp,
        'FN (fraud missed)': fn,
        'FP (false alarms)': fp,
    })

final_df = pd.DataFrame(final_rows)
final_df.set_index('Model', inplace=True)

# Print in sections for readability
print('Cross-Validation Performance (5-fold, from scratch):')
print(final_df[['CV AUC (mean±std)', 'CV AP (mean±std)', 'CV F1 (mean±std)']].to_string())

print('\nTest Set Performance (holdout 20%):')
print(final_df[['Test Accuracy', 'Test AUC-ROC', 'Test AP', 'Test F1',
                'TP (fraud caught)', 'FN (fraud missed)', 'FP (false alarms)']].to_string())

In [ ]:
# ── Model Recommendation ──────────────────────────────────────────────────────
# Structured business-facing recommendation based on evidence

print('MODEL RECOMMENDATION MEMO')
print('=' * 80)
print()

# Identify the winner by AUC (ranking quality) and AP (imbalanced class)
best_auc = max(metrics_all, key=lambda n: metrics_all[n]['AUC-ROC'])
best_ap  = max(metrics_all, key=lambda n: metrics_all[n]['AP (Avg Precision)'])
best_f1  = max(metrics_all, key=lambda n: metrics_all[n]['F1'])

# Check if differences are statistically significant
sig_pairs = mcnemar_df[mcnemar_df['Sig_Bonferroni']]

print(f'Best by AUC-ROC : {best_auc}  ({metrics_all[best_auc]["AUC-ROC"]:.4f})')
print(f'Best by AP      : {best_ap}   ({metrics_all[best_ap]["AP (Avg Precision)"]:.4f})')
print(f'Best by F1      : {best_f1}   ({metrics_all[best_f1]["F1"]:.4f})')
print()
print(f'Statistical significance (Bonferroni-corrected, α={alpha_bonf:.4f}):')
if len(sig_pairs) > 0:
    for _, row in sig_pairs.iterrows():
        print(f'  {row["Model A"]} vs {row["Model B"]}: p={row["p_value"]:.6f} → significant')
else:
    print('  No pairs are significant after Bonferroni correction.')
    print('  Interpretation: Models perform similarly; the observed differences may be noise.')

print()
print('RECOMMENDATION:')
print('-' * 60)
print(f'''
For a 3% fraud rate environment, we recommend {best_ap} as the primary
production model for the following reasons:

1. METRIC CHOICE: With extreme class imbalance (3% fraud), Average Precision
   (AP) is more informative than AUC-ROC. AUC can be high even when the model
   performs poorly on the minority class. AP directly measures the precision-
   recall trade-off that matters for fraud.

2. CALIBRATION: Logistic Regression produces well-calibrated probabilities,
   enabling reliable threshold selection. Threshold tuning is critical for
   optimising the precision/recall trade-off for each business context.

3. ROBUSTNESS: CV scores (mean ± std) show consistent performance across folds
   with low variance, indicating the model generalises well.

4. INTERPRETABILITY: Logistic Regression coefficients can be inspected and
   explained to regulators and fraud analysts.

5. THRESHOLD RECOMMENDATION: Deploy with threshold ≈ {optimal_thresholds.get(best_ap, 0.5):.3f}
   (optimal F1 point) rather than default 0.5.

CAVEATS:
- Differences between top models are not statistically significant after
  Bonferroni correction. Monitor performance on live traffic.
- Consider re-evaluating on a larger holdout set before final deployment.
- If recall=80% is a hard regulatory requirement, verify the operating threshold.
''')

## Section 9: Self-Check Questions

Five questions covering all major concepts from this lab. Attempt each before expanding the answer.

---

**Q1.** Your 5-fold CV shows:
- LogReg accuracy: `[0.97, 0.97, 0.98, 0.97, 0.97]`
- DecTree accuracy: `[0.97, 0.65, 0.98, 0.97, 0.97]`

Which model is more reliable and why?

<details>
<summary>Show Answer</summary>

**LogReg is far more reliable.** Although both models have the same mean accuracy (~0.97 when averaged including the outlier fold), DecisionTree has a catastrophic fold where it scored only 0.65 — meaning on that 20% slice of data it barely beat random guessing. This high variance across folds (std ≈ 0.13 for DecTree vs ≈ 0.004 for LogReg) indicates the Decision Tree is **overfitting** to whichever training split it sees. It memorises patterns that don't generalise. A model with mean=0.93 and std=0.002 is far preferable to mean=0.94 and std=0.13 in production — the latter is unpredictable. Always report mean AND standard deviation for CV scores.

</details>

---

**Q2.** ROC-AUC says k-NN wins; Average Precision says LogReg wins. Which metric should you trust for 3% fraud rate?

<details>
<summary>Show Answer</summary>

**Trust Average Precision (AP) for the 3% fraud rate.** Here is why:

ROC-AUC measures the model's ability to rank a randomly chosen positive above a randomly chosen negative. With 97% legitimate transactions, the vast majority of random pairs are (legit, fraud), and the ROC curve is dominated by the model's performance on the majority class. A model can achieve high AUC-ROC while performing poorly on fraud cases specifically.

Average Precision directly measures the precision-recall trade-off as you vary the classification threshold. It gives equal weight to every recall level and penalises models that produce many false positives at any operating point. For imbalanced datasets (like fraud), AP is the standard recommendation because it is sensitive to the minority class performance — which is exactly what we care about.

Rule: use AUC-ROC for roughly balanced classes; use AP (or PR-AUC) for severe class imbalance.

</details>

---

**Q3.** McNemar's test gives p = 0.04 between LogReg and k-NN. The Bonferroni-corrected threshold is 0.0083. Is the difference significant?

<details>
<summary>Show Answer</summary>

**No — not after correction.** The raw p-value (0.04) is below α = 0.05, so we would call it significant without correction. However, we ran 6 tests (C(4,2) = 6 pairs), so we apply Bonferroni: α_corrected = 0.05/6 = 0.0083. Since p = 0.04 > 0.0083, we **fail to reject H₀** after correction.

Interpretation: the difference between LogReg and k-NN is plausibly due to chance given that we ran 6 simultaneous tests. We cannot conclude one is definitively better based on this dataset alone. Options: (1) collect more data to increase statistical power, (2) use a less conservative correction like Benjamini-Hochberg FDR, or (3) run a 5×2 CV paired t-test for a more robust estimate.

</details>

---

**Q4.** The calibration plot shows GaussianNB predicts P(fraud) = 0.9 for cases that are actually only 40% fraud. What problem does this cause for threshold selection?

<details>
<summary>Show Answer</summary>

**It breaks threshold selection.** If you naively set threshold = 0.5 to "flag likely fraud", GaussianNB would flag only the cases it assigns ≥0.9, but those cases are actually only 40% fraud. You would be using the wrong part of the probability scale.

More concretely: a fraud analyst using this model cannot say "flag anything above P=0.3 because that's where 30% fraud rate begins" — the model's probabilities don't correspond to actual fraud rates. They would need to reverse-engineer the miscalibration to find the threshold corresponding to their desired operating point.

The fix: apply **Platt scaling** (logistic calibration) or **isotonic regression** post-hoc to correct the probability outputs. After calibration, P=0.4 would genuinely mean 40% of flagged cases are fraud, making threshold selection interpretable and reliable.

</details>

---

**Q5.** You achieve recall = 0.80 at threshold = 0.3. At that threshold, precision = 0.12. How many legitimate transactions are blocked for every 1 fraudster caught?

<details>
<summary>Show Answer</summary>

**Approximately 7.3 legitimate transactions blocked per fraudster caught.**

Precision = TP / (TP + FP) = 0.12

This means: of all transactions flagged as fraud, 12% are actual fraudsters and 88% are legitimate.

Ratio = FP / TP = (1 - Precision) / Precision = 0.88 / 0.12 ≈ **7.33**

So for every fraudster you catch, you also block approximately 7.3 legitimate customers. This is the precision-recall trade-off in action: pushing recall to 0.80 at a 3% base rate inevitably creates many false positives. Whether 7.3:1 is acceptable depends entirely on business context — the cost of fraud vs the cost of customer friction and potential churn from declined legitimate transactions.

</details>